### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


In [1]:
!pip install langchain
!pip install langchain-chroma
!pip install langchain_groq

### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [3]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

groq_api_key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api_key)

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

huggingfacembeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12032.79it/s]


In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents,embedding=huggingfacembeddings)
vectorstore

In [8]:
vectorstore.similarity_search("cat")

[Document(id='07564c65-a76e-41dd-a728-11409baa9bd7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='55d01e17-8457-482c-85c7-5f282a2d89f9', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='58d53579-498b-4e74-9009-7142554664df', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='d84155f2-e62a-4be7-ae94-d1eb2bf12091', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [9]:
#Async query
await vectorstore.asimilarity_search("cat")

[Document(id='07564c65-a76e-41dd-a728-11409baa9bd7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='55d01e17-8457-482c-85c7-5f282a2d89f9', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='58d53579-498b-4e74-9009-7142554664df', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='d84155f2-e62a-4be7-ae94-d1eb2bf12091', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [10]:
#search with similarity score

vectorstore.similarity_search_with_score("cat")

[(Document(id='07564c65-a76e-41dd-a728-11409baa9bd7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056218147278),
 (Document(id='55d01e17-8457-482c-85c7-5f282a2d89f9', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='58d53579-498b-4e74-9009-7142554664df', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956907272338867),
 (Document(id='d84155f2-e62a-4be7-ae94-d1eb2bf12091', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.665792465209961)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [15]:
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='07564c65-a76e-41dd-a728-11409baa9bd7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='55d01e17-8457-482c-85c7-5f282a2d89f9', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [17]:
#with as_retriever

retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":1})
retriever.batch(["cat","dog"])

[[Document(id='07564c65-a76e-41dd-a728-11409baa9bd7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='55d01e17-8457-482c-85c7-5f282a2d89f9', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [21]:
## RAG

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

message = """
Answer the following questions
{question}

context
{context}
"""

prompt = ChatPromptTemplate.from_messages(["human",message])
rag_chain = {"context":retriever,"question":RunnablePassthrough()}|prompt|model

rag_chain.invoke("tell me about cats?")

AIMessage(content='Cats are independent pets that often enjoy their own space.\u202fThey tend to be self‑reliant, preferring to have a comfortable spot where they can relax and observe their surroundings rather than seeking constant attention. This temperament makes them well‑suited to owners who appreciate a companion that can entertain itself while still being affectionate on its own terms.\u202f【07564c65-a76e-41dd-a728-11409baa9bd7】', additional_kwargs={'reasoning_content': 'We need to answer the user question: "Answer the following questions tell me about cats?" There\'s a context document. We should use the provided document. Also follow policies: no disallowed content. It\'s straightforward. Provide info about cats based on document, maybe expand with general knowledge but staying relevant. Use citations. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 167, 'prompt_tokens': 140, 'total_tokens': 307, 'completion_time': 0.351668432, 'completion_tokens_det